# Практика · Тема 13. AlexNet і VGG

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) · ДЗ: [homework.md](homework.md)

**Мережа не потрібна:** архітектури будуються локально, ваги не завантажуються.
Виклик `torchvision.models.vgg16(weights=None)` створює всі шари й ініціалізує їх
випадковими числами — жодного байта з інтернету.

⏱ Зошит будує кілька справжніх архітектур, міряє час прямого проходу й один раз
навчає маленьку мережу на синтетичних фігурах. Заміряно: **близько двох хвилин**
на чотирьох ядрах без відеокарти.

Що зробимо:

1. побудуємо AlexNet, VGG16 і ResNet18 без ваг і порахуємо, де в них сидять параметри;
2. розберемо обидві мережі теми **по шарах** — форми, ядра, кроки, параметри;
3. порахуємо параметри одного шару **руками** і звіримо з `sum(p.numel())`;
4. зміряємо памʼять на ваги й на активації та час прямого проходу;
5. порахуємо **рецептивне поле** ядра 11×11 проти стосу 3×3;
6. перевіримо, скільки економлять два 3×3 замість одного 5×5 на масштабі VGG;
7. подивимось на групи в згортках і на те, що від двох гілок AlexNet лишилось;
8. навчимо маленьку VGG-подібну мережу на наших фігурах 28×28.

## 0. Налаштування

Перша клітинка ставить `torch.set_num_threads(1)`. Причин дві, і жодна не про лінощі:

- на маленьких тензорах багатопотоковість не пришвидшує, а сповільнює — координація
  потоків коштує більше, ніж сама арифметика;
- під кількома потоками додавання чисел із рухомою комою йде в іншому порядку, і суми
  виходять трохи різні від прогону до прогону. З одним потоком числа відтворюються.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn

# один потік: і швидше на малих тензорах, і числа відтворюються від прогону до прогону
torch.set_num_threads(1)

import torchvision
from torchvision import models

print("torch      :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("потоків    :", torch.get_num_threads())

## 1. Будуємо архітектури без жодного завантаження

`weights=None` — це не «випадкові ваги замість справжніх», а «взагалі не чіпай мережу».
Функція збирає шари за описом архітектури й ініціалізує їх звичайним випадковим
розподілом. Для того, що ми рахуємо — форм, кількості параметрів, часу проходу —
значення ваг не має значення взагалі.

In [ ]:
# будуємо три мережі локально: жодного звернення до мережі не відбувається
alexnet = models.alexnet(weights=None)
vgg16 = models.vgg16(weights=None)
resnet18 = models.resnet18(weights=None)

for name, model in [("AlexNet", alexnet), ("VGG16", vgg16), ("ResNet18", resnet18)]:
    total = sum(p.numel() for p in model.parameters())
    print(f"{name:<10} параметрів: {total:>12,}")

## 2. Головна таблиця теми: де сидять ваги

Ділимо параметри на дві купи — ті, що в згорткових шарах, і ті, що в повнозвʼязних.
Саме це співвідношення і є сюжетом теми.

In [ ]:
def weight_split(model):
    """Ділить параметри моделі на згорткові й повнозвʼязні.

    recurse=False обовʼязково: інакше параметри вкладених модулів порахуються двічі.
    """
    conv = sum(p.numel() for m in model.modules() if isinstance(m, nn.Conv2d)
               for p in m.parameters(recurse=False))
    linear = sum(p.numel() for m in model.modules() if isinstance(m, nn.Linear)
                 for p in m.parameters(recurse=False))
    total = sum(p.numel() for p in model.parameters())
    conv_layers = sum(1 for m in model.modules() if isinstance(m, nn.Conv2d))
    return total, conv, linear, conv_layers


print(f"{'мережа':<10} {'параметрів':>13} {'у згортках':>12} {'у лінійних':>12} {'згорток':>9}")
print("-" * 60)
for name, model in [("AlexNet", alexnet), ("VGG16", vgg16), ("ResNet18", resnet18)]:
    total, conv, linear, conv_layers = weight_split(model)
    print(f"{name:<10} {total:>13,} {100*conv/total:>11.1f}% {100*linear/total:>11.1f}% {conv_layers:>9}")

Прочитай третій рядок разом із першими двома. ResNet18 має **більше** згорткових
шарів, ніж VGG16, і при цьому важить у дванадцять разів менше. Різниця не в глибині,
а в голові — і саме її ми зараз розберемо по шарах.

## 3. AlexNet по шарах

Проганяємо порожній тензор потрібної форми крізь мережу й на кожному шарі друкуємо, що
з ним сталося. Це найчесніший спосіб розбору: форми не вгадуються з формул, а беруться
з самого `torch`.

In [ ]:
def layer_report(model, name, input_shape=(1, 3, 224, 224)):
    """Друкує пошаровий звіт: форма входу, форма виходу, параметри, частка ваг.

    Повертає список рядків, щоб далі рахувати з них памʼять.
    """
    x = torch.zeros(*input_shape)
    total = sum(p.numel() for p in model.parameters())
    rows = []
    print(f"=== {name}: вхід {tuple(input_shape)} ===")
    print(f"{'шар':<12}{'опис':<26}{'вихід':<20}{'параметрів':>13}{'частка':>9}")
    print("-" * 82)

    def describe(module):
        if isinstance(module, nn.Conv2d):
            return (f"{module.in_channels}→{module.out_channels}, "
                    f"{module.kernel_size[0]}×{module.kernel_size[0]}, крок {module.stride[0]}")
        if isinstance(module, nn.MaxPool2d):
            return f"max {module.kernel_size}×{module.kernel_size}, крок {module.stride}"
        if isinstance(module, nn.Linear):
            return f"{module.in_features}→{module.out_features}"
        return ""

    with torch.no_grad():
        for i, module in enumerate(model.features):
            x = module(x)
            if isinstance(module, (nn.Conv2d, nn.MaxPool2d)):
                params = sum(p.numel() for p in module.parameters())
                rows.append((f"f{i}", type(module).__name__, tuple(x.shape)[1:], x.numel(), params))
                print(f"{'f'+str(i):<12}{describe(module):<26}{str(tuple(x.shape)[1:]):<20}"
                      f"{params:>13,}{100*params/total:>8.2f}%")
        x = model.avgpool(x)
        x = torch.flatten(x, 1)
        print(f"{'Flatten':<12}{'розпластати карту':<26}{str(tuple(x.shape)[1:]):<20}{0:>13,}{0:>8.2f}%")
        for i, module in enumerate(model.classifier):
            x = module(x)
            if isinstance(module, nn.Linear):
                params = sum(p.numel() for p in module.parameters())
                rows.append((f"c{i}", "Linear", tuple(x.shape)[1:], x.numel(), params))
                print(f"{'c'+str(i):<12}{describe(module):<26}{str(tuple(x.shape)[1:]):<20}"
                      f"{params:>13,}{100*params/total:>8.2f}%")
    print("-" * 82)
    print(f"{'усього':<58}{total:>13,}{100.0:>8.1f}%")
    return rows


alexnet_rows = layer_report(alexnet, "AlexNet")

Три останні рядки — це 96 % мережі. Пʼять згорток, які власне й дивляться на
зображення, разом дають 4 %.

Тепер VGG16. Мережа довша, зате однорідна: тільки ядра 3×3 і тільки пулінги 2×2.

In [ ]:
vgg_rows = layer_report(vgg16, "VGG16")

## 4. Рахуємо параметри руками — і звіряємо з бібліотекою

Це головна перевірка практики: усередині `torch` немає магії. Формула параметрів
згортки виведена в темі 07:

    параметрів = c_вх × c_вих × k × k + c_вих

Останній доданок — по одному зсуву на кожен вихідний канал. Для повнозвʼязного шару
формула ще простіша: кожен вхід зʼєднаний із кожним виходом, плюс зсуви:

    параметрів = вхід × вихід + вихід

In [ ]:
def conv_params_by_hand(conv):
    """Наша формула параметрів згортки — без жодного звернення до .parameters()."""
    kh, kw = conv.kernel_size
    # groups ділить вхідні канали між групами: кожна вихідна карта бачить лише свою частину
    weights = (conv.in_channels // conv.groups) * conv.out_channels * kh * kw
    biases = conv.out_channels if conv.bias is not None else 0
    return weights + biases


def linear_params_by_hand(linear):
    """Те саме для повнозвʼязного шару."""
    biases = linear.out_features if linear.bias is not None else 0
    return linear.in_features * linear.out_features + biases


checks = [
    ("AlexNet conv1", alexnet.features[0], conv_params_by_hand),
    ("AlexNet conv2", alexnet.features[3], conv_params_by_hand),
    ("VGG16 conv5_3", vgg16.features[28], conv_params_by_hand),
    ("VGG16 Linear 25088→4096", vgg16.classifier[0], linear_params_by_hand),
    ("AlexNet Linear 9216→4096", alexnet.classifier[1], linear_params_by_hand),
]

for label, layer, formula in checks:
    by_hand = formula(layer)
    by_torch = sum(p.numel() for p in layer.parameters())
    print(f"{label:<26} руками {by_hand:>12,}   torch {by_torch:>12,}")
    assert by_hand == by_torch, f"розрахунок розійшовся на шарі {label}!"

print()
print("✅ збігається: усі пʼять шарів порахувалися руками так само, як у torch")

## 5. Звідки береться найдорожчий шар

Найдорожчий шар VGG16 — перший `Linear` після `Flatten`. Порахуймо його ланцюжок
з нуля: пʼять пулінгів ділять сторону навпіл, канали доходять до 512, `Flatten`
перетворює карту на рядок, і цей рядок множиться на 4096 виходів.

In [ ]:
side = 224
for step in range(5):
    side //= 2          # кожен max-пулінг 2×2 із кроком 2 ділить сторону навпіл
    print(f"після пулінгу {step+1}: сторона {side}")

channels = 512
flat_length = channels * side * side
print()
print(f"карта перед головою: {channels}×{side}×{side}")
print(f"довжина рядка після Flatten: {channels} × {side} × {side} = {flat_length:,}")

first_linear = flat_length * 4096 + 4096
print(f"перший Linear: {flat_length:,} × 4096 + 4096 = {first_linear:,}")

conv_part = sum(p.numel() for m in vgg16.modules() if isinstance(m, nn.Conv2d)
                for p in m.parameters(recurse=False))
print()
print(f"уся згорткова частина VGG16: {conv_part:,}")
print(f"один цей шар важчий за неї у {first_linear/conv_part:.1f} раза")

### Ціна голови від розміру входу

Довжина рядка після `Flatten` пропорційна **квадрату** сторони входу. Тому ціна першого
`Linear` росте квадратично, а голова з глобальним усередненням не росте взагалі: вона
завжди бере по одному числу з кожного каналу.

In [ ]:
gap_head = 512 * 1000 + 1000          # GAP → Linear(512 → 1000 класів)

print(f"{'вхід':>9} {'карта':>14} {'рядок':>9} {'ваг у Linear':>15} {'МБ':>7} {'проти GAP':>11}")
print("-" * 70)
for input_side in (128, 160, 192, 224, 256, 320, 384, 448):
    map_side = input_side // 32       # пʼять пулінгів = ділення на 32
    row = 512 * map_side * map_side
    params = row * 4096 + 4096
    print(f"{str(input_side)+'×'+str(input_side):>9} {f'512×{map_side}×{map_side}':>14} "
          f"{row:>9,} {params:>15,} {params*4/1024/1024:>7.0f} {params/gap_head:>10.0f}×")

print()
print(f"голова на глобальному усередненні: {gap_head:,} ваг за будь-якого розміру входу")

### А скільки б важила VGG16 без голови

Замінімо три повнозвʼязні шари на глобальне усереднення й один `Linear` на тисячу
класів. Згорткову частину не чіпаємо взагалі.

In [ ]:
vgg_total = sum(p.numel() for p in vgg16.parameters())
vgg_head = vgg_total - conv_part
vgg_with_gap = conv_part + gap_head

print(f"VGG16 як є            : {vgg_total:>12,}")
print(f"  з них голова        : {vgg_head:>12,}  ({100*vgg_head/vgg_total:.1f} %)")
print(f"VGG16 з GAP-головою   : {vgg_with_gap:>12,}")
print()
print(f"схудла б у {vgg_total/vgg_with_gap:.1f} раза — при тій самій згортковій частині")

## 6. Памʼять: ваги проти активацій

**Ваги** — це параметри моделі, вони лежать у памʼяті постійно. **Активації** — це
проміжні карти ознак, які живуть лише під час проходу, зате множаться на розмір батча.

Порахуймо обидві величини й знайдемо шар-рекордсмен у кожній.

In [ ]:
def memory_report(rows, name, total_params):
    """Ваги й активації по шарах: скільки мегабайт і де максимум."""
    act_elements = sum(r[3] for r in rows)
    heaviest_weight = max(rows, key=lambda r: r[4])
    heaviest_act = max(rows, key=lambda r: r[3])
    print(f"=== {name} ===")
    print(f"ваги            : {total_params:>12,} чисел = {total_params*4/1024/1024:>8.2f} МБ")
    print(f"активації, батч 1: {act_elements:>12,} чисел = {act_elements*4/1024/1024:>8.2f} МБ")
    print(f"найважчий за вагами     : {heaviest_weight[0]} — "
          f"{heaviest_weight[4]:,} ваг ({heaviest_weight[4]*4/1024/1024:.2f} МБ)")
    print(f"найважчий за активаціями: {heaviest_act[0]} — форма {heaviest_act[2]}, "
          f"{heaviest_act[3]:,} чисел ({heaviest_act[3]*4/1024/1024:.2f} МБ)")
    # з якого батча активації переважать ваги
    print(f"активації переважать ваги на батчі {total_params/act_elements:.0f}")
    print()
    return act_elements


alex_acts = memory_report(alexnet_rows, "AlexNet",
                          sum(p.numel() for p in alexnet.parameters()))
vgg_acts = memory_report(vgg_rows, "VGG16", vgg_total)

Ось повторення відкриття з теми 09, тільки на справжніх мережах: **максимум ваг і
максимум памʼяті стоять у різних кінцях мережі**. Найдорожчий за вагами шар VGG16 —
перший `Linear`; найдорожчий за памʼяттю — найперша згортка, у якій ваг усього 1 792.

## 7. Час прямого проходу

Міряємо на одному зображенні, в один потік, у режимі `eval()` і без градієнтів.
Беремо **мінімум** із кількох прогонів, а не середнє: мінімум найменше залежить від
того, чим ще зайнята машина.

⚠️ Числа плаватимуть. У мене вони від прогону до прогону відрізнялись у два-три рази.
Дивись не на самі числа, а на **порядок величини** й на співвідношення між мережами.

In [ ]:
def conv_block(in_channels, out_channels):
    """Той самий блок Conv → ReLU → Pool, що в темі 09."""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
    )


class ShapeNet(nn.Module):
    """Наша маленька мережа з теми 09: три блоки й голова з Flatten."""

    def __init__(self, num_classes=6):
        super().__init__()
        self.body = nn.Sequential(conv_block(1, 8), conv_block(8, 16), conv_block(16, 32))
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 3 * 3, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.head(self.body(x))


def forward_time_ms(model, sample, runs=10):
    """Мінімальний час прямого проходу в мілісекундах."""
    model.eval()
    with torch.no_grad():
        model(sample)                      # прогрів: перший прохід виділяє буфери
        times = []
        for _ in range(runs):
            started = time.perf_counter()
            model(sample)
            times.append((time.perf_counter() - started) * 1000)
    return min(times)


shapenet = ShapeNet()
big_sample = torch.zeros(1, 3, 224, 224)
small_sample = torch.zeros(1, 1, 28, 28)

print(f"{'мережа':<12}{'вхід':<16}{'параметрів':>13}{'час, мс':>10}")
print("-" * 51)
for name, model, sample in [("ShapeNet", shapenet, small_sample),
                            ("AlexNet", alexnet, big_sample),
                            ("VGG16", vgg16, big_sample),
                            ("ResNet18", resnet18, big_sample)]:
    params = sum(p.numel() for p in model.parameters())
    print(f"{name:<12}{str(tuple(sample.shape)[1:]):<16}{params:>13,}"
          f"{forward_time_ms(model, sample):>10.2f}")

VGG16 важить лише вдвічі більше за AlexNet, а рахується приблизно вдесятеро довше.
Причина в тому, що час іде не на ваги, а на множення: AlexNet у першому ж шарі стискає
картинку вчетверо кроком 4, а VGG16 чесно проганяє дві згортки по стороні 224.

Тобто **параметри й обчислення — різні ресурси**. Далі ми це порахуємо прямо.

## 8. Рецептивне поле: 11×11 з кроком 4 проти стосу 3×3

**Рецептивне поле** — скільки пікселів вхідного зображення бачить один нейрон карти.
Формула накопичувальна: кожен шар додає до поля `(k − 1)`, помножене на **сумарний
крок усіх попередніх шарів**.

    поле_нове = поле_старе + (k − 1) × крок_накопичений
    крок_накопичений = крок_накопичений × s

In [ ]:
def receptive_field(layers, title):
    """Рецептивне поле й накопичений крок після кожного шару.

    layers — список (назва, розмір ядра, крок).
    """
    field, jump = 1, 1
    print(f"--- {title} ---")
    for name, kernel, stride in layers:
        field = field + (kernel - 1) * jump
        jump = jump * stride
        print(f"  після {name:<18} поле {field:>4}×{field:<4} крок {jump}")
    return field, jump


big_field, big_jump = receptive_field([("conv1 11×11 крок 4", 11, 4)],
                                      "AlexNet, лише перший шар")
stack_field, stack_jump = receptive_field([(f"conv 3×3 №{i}", 3, 1) for i in range(1, 6)],
                                          "стос із пʼяти 3×3, крок 1")

print()
print(f"поле однакове: {big_field} проти {stack_field}")
print(f"але крок різний: {big_jump} проти {stack_jump} — грубе ядро одразу зменшує карту вчетверо")

Поле однакове — тепер порівняймо ціну. Рахуємо параметри й **кількість множень** на
вході 224×224. Множення — це те, за що платить процесор.

In [ ]:
def conv_cost(in_channels, out_channels, kernel, output_side):
    """Параметри й множення одного згорткового шару."""
    params = in_channels * out_channels * kernel * kernel + out_channels
    # на кожну точку вихідної карти й кожен вихідний канал припадає cin × k × k множень
    mults = output_side * output_side * out_channels * in_channels * kernel * kernel
    return params, mults


# варіант AlexNet: одне ядро 11×11 із кроком 4 дає карту 55×55
big_params, big_mults = conv_cost(3, 64, 11, 55)

# варіант «як у сучасній мережі»: пʼять шарів 3×3 із кроком 1, сторона лишається 224
stack_params, stack_mults = conv_cost(3, 64, 3, 224)
for _ in range(4):
    params, mults = conv_cost(64, 64, 3, 224)
    stack_params += params
    stack_mults += mults

print(f"{'варіант':<26}{'поле':>6}{'параметрів':>13}{'множень':>18}")
print("-" * 63)
print(f"{'одна 11×11, крок 4':<26}{big_field:>6}{big_params:>13,}{big_mults:>18,}")
print(f"{'пʼять 3×3, крок 1':<26}{stack_field:>6}{stack_params:>13,}{stack_mults:>18,}")
print()
print(f"стос дорожчий за вагами у {stack_params/big_params:.1f} раза")
print(f"стос дорожчий за множеннями у {stack_mults/big_mults:.0f} разів")

Ось відповідь на питання «чому у 2012 році робили так». Не через красу, а тому, що
сто разів більше множень на двох відеокартах того часу означало не «повільніше», а
«неможливо».

Для повноти — рецептивне поле цілих мереж. Останній шар AlexNet бачить майже все
зображення, і VGG16 теж.

In [ ]:
alexnet_layers = [
    ("conv1 11×11 s4", 11, 4), ("pool 3×3 s2", 3, 2),
    ("conv2 5×5", 5, 1),       ("pool 3×3 s2", 3, 2),
    ("conv3 3×3", 3, 1), ("conv4 3×3", 3, 1), ("conv5 3×3", 3, 1),
    ("pool 3×3 s2", 3, 2),
]
receptive_field(alexnet_layers, "AlexNet повністю")

vgg_layers = []
for block, repeats in enumerate([2, 2, 3, 3, 3], start=1):
    for i in range(repeats):
        vgg_layers.append((f"conv{block}_{i+1} 3×3", 3, 1))
    vgg_layers.append((f"pool{block} 2×2", 2, 2))
field, jump = receptive_field(vgg_layers, "VGG16 повністю")
print()
print(f"останній шар VGG16 бачить квадрат {field}×{field} — більший за саме зображення 224×224")

## 9. Два 3×3 замість одного 5×5 — на масштабі VGG

Тема 07 показала це на одному шарі. Перевіримо, що відношення тримається на всіх
масштабах каналів, а потім порахуємо, скільки коштувала б уся згорткова частина
VGG16 на ядрах 5×5.

In [ ]:
def conv_weight_count(in_channels, out_channels, kernel):
    return in_channels * out_channels * kernel * kernel + out_channels


print(f"{'канали':<14}{'одна 5×5':>12}{'дві 3×3':>12}{'дешевше на':>13}")
print("-" * 51)
for channels in (64, 128, 256, 512):
    one_5x5 = conv_weight_count(channels, channels, 5)
    two_3x3 = 2 * conv_weight_count(channels, channels, 3)
    print(f"{f'{channels} → {channels}':<14}{one_5x5:>12,}{two_3x3:>12,}"
          f"{100*(1-two_3x3/one_5x5):>12.1f}%")

In [ ]:
# план каналів VGG16: пари «скільки входить → скільки виходить» для всіх 13 згорток
vgg_plan = [(3, 64), (64, 64), (64, 128), (128, 128),
            (128, 256), (256, 256), (256, 256),
            (256, 512), (512, 512), (512, 512),
            (512, 512), (512, 512), (512, 512)]

as_3x3 = sum(conv_weight_count(cin, cout, 3) for cin, cout in vgg_plan)
as_5x5 = sum(conv_weight_count(cin, cout, 5) for cin, cout in vgg_plan)

print(f"згорткова частина VGG16, ядра 3×3: {as_3x3:>12,}")
print(f"та сама структура, ядра 5×5      : {as_5x5:>12,}")
print(f"різниця                          : {as_5x5-as_3x3:>+12,}  (у {as_5x5/as_3x3:.2f} раза)")
print()
# перевіряємо, що наш рахунок збігається зі справжньою мережею
assert as_3x3 == conv_part, "план каналів розійшовся зі справжньою VGG16!"
print("✅ наш план каналів дав рівно стільки ж, скільки справжня VGG16:", f"{conv_part:,}")
print()
print(f"множник {as_5x5/as_3x3:.2f} — це просто 25/9 = {25/9:.2f}, відношення площ ядер")

## 10. Дві гілки AlexNet і групована згортка

Оригінальна AlexNet була розрізана навпіл, бо не влізала у три гігабайти відеопамʼяті
GTX 580. У сучасних термінах це **групована згортка** з двома групами: кожна вихідна
карта підключена лише до половини вхідних каналів.

Спершу подивимось, що лишилось від цього у версії `torchvision`.

In [ ]:
groups_in_torchvision = [m.groups for m in alexnet.modules() if isinstance(m, nn.Conv2d)]
print("groups у пʼяти згортках torchvision-версії AlexNet:", groups_in_torchvision)
print("тобто гілок немає: мережу давно склеїли назад в одну")
print()

# оригінальна друга згортка: 96 вхідних каналів, 256 вихідних, ядро 5×5
without_groups = 96 * 256 * 5 * 5 + 256
with_two_groups = (96 // 2) * 256 * 5 * 5 + 256
print(f"conv2 без груп : {without_groups:,}")
print(f"conv2 з groups=2: {with_two_groups:,}")
print(f"економія        : {without_groups-with_two_groups:,} — рівно половина ваг")

In [ ]:
# збираємо оригінальну AlexNet 2012 року за описом статті — лише щоб порахувати ваги
original_alexnet = [
    ("conv1 3→96, 11×11", 3, 96, 11, 1),
    ("conv2 96→256, 5×5", 96, 256, 5, 2),
    ("conv3 256→384, 3×3", 256, 384, 3, 1),
    ("conv4 384→384, 3×3", 384, 384, 3, 2),
    ("conv5 384→256, 3×3", 384, 256, 3, 2),
]

grouped_total = 0
plain_total = 0
for label, cin, cout, kernel, groups in original_alexnet:
    grouped = (cin // groups) * cout * kernel * kernel + cout
    plain = cin * cout * kernel * kernel + cout
    grouped_total += grouped
    plain_total += plain
    mark = "  ← дві гілки" if groups == 2 else ""
    print(f"{label:<22} з групами {grouped:>10,}   без груп {plain:>10,}{mark}")

head = (9216 * 4096 + 4096) + (4096 * 4096 + 4096) + (4096 * 1000 + 1000)
print()
print(f"голова (три Linear)    : {head:>12,}")
print(f"оригінал 2012 разом    : {grouped_total + head:>12,}")
print(f"та сама мережа без груп: {plain_total + head:>12,}")
print(f"дві гілки зекономили   : {plain_total - grouped_total:>12,} параметрів")

Порядок подій варто памʼятати саме такий: **спершу обхідний маневр, і тільки потім
теорія**. Групи зʼявились не тому, що хтось довів їхню користь, а тому, що інакше
мережа не запускалась. Осмислили їх пізніше — і на цьому побудували економні
архітектури, про які буде окрема тема.

## 11. Маленька VGG-подібна мережа на наших фігурах

Справжню VGG16 ми не навчаємо: на 28×28 вона безглузда, а на 224×224 один прохід іде
пів секунди, тож епоха забрала б години. Зате ідею VGG можна перевірити на своєму
масштабі — зберемо мережу за трьома правилами VGG (тільки 3×3, тільки пулінг 2×2,
подвоєння каналів) і навчимо її розрізняти шість фігур.

Датасет той самий, що в блоці 2: фігури 28×28, згенеровані формулами.

In [ ]:
def make_shapes(n_samples, seed):
    """Генерує зображення шести фігур 28×28. Нічого не завантажується."""
    rng = np.random.default_rng(seed)
    images = np.zeros((n_samples, 1, 28, 28), dtype=np.float32)
    labels = rng.integers(0, 6, size=n_samples)
    grid_y, grid_x = np.mgrid[0:28, 0:28]

    for i in range(n_samples):
        # центр гуляє, щоб мережа не вивчила «фігура завжди посередині»
        center_y = 14 + rng.integers(-4, 5)
        center_x = 14 + rng.integers(-4, 5)
        radius = rng.integers(5, 9)
        dy = grid_y - center_y
        dx = grid_x - center_x
        shape = labels[i]

        if shape == 0:                                    # коло
            mask = dy**2 + dx**2 <= radius**2
        elif shape == 1:                                  # квадрат
            mask = (np.abs(dy) <= radius) & (np.abs(dx) <= radius)
        elif shape == 2:                                  # ромб
            mask = np.abs(dy) + np.abs(dx) <= radius
        elif shape == 3:                                  # кільце
            distance = np.sqrt(dy**2 + dx**2)
            mask = (distance <= radius) & (distance >= radius - 2)
        elif shape == 4:                                  # хрест
            mask = ((np.abs(dy) <= 1) & (np.abs(dx) <= radius)) | \
                   ((np.abs(dx) <= 1) & (np.abs(dy) <= radius))
        else:                                             # трикутник
            mask = (dy <= radius) & (dy >= -radius) & (np.abs(dx) <= (radius - dy) / 2)

        images[i, 0][mask] = 1.0
        images[i, 0] += rng.normal(0, 0.20, size=(28, 28)).astype(np.float32)

    return torch.from_numpy(images), torch.from_numpy(labels.astype(np.int64))


train_x, train_y = make_shapes(1200, seed=42)
test_x, test_y = make_shapes(300, seed=7)

# нормалізуємо вхід статистикою навчальної вибірки — тема 11 пояснювала, навіщо:
# без цього робоче вікно швидкості навчання зсувається разом із масштабом даних
pixel_mean = train_x.mean()
pixel_std = train_x.std()
train_x = (train_x - pixel_mean) / pixel_std
test_x = (test_x - pixel_mean) / pixel_std

print("навчальних:", tuple(train_x.shape), "перевірочних:", tuple(test_x.shape))
print("класів:", int(train_y.max()) + 1)
print(f"вхід нормалізовано: середнє {pixel_mean:.3f}, відхилення {pixel_std:.3f}")

In [ ]:
def vgg_block(in_channels, out_channels, convs):
    """Блок у стилі VGG: кілька згорток 3×3 поспіль, потім один пулінг 2×2.

    Батч-нормалізація з теми 11 стоїть тут не для краси: шість згорток поспіль —
    уже досить глибоко, щоб звичайний SGD без нормалізації буксував на рівні
    вгадування. У самої VGG батчнорму не було — його придумали через рік.
    """
    layers = []
    channels = in_channels
    for _ in range(convs):
        layers.append(nn.Conv2d(channels, out_channels, kernel_size=3, padding=1))
        layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU())
        channels = out_channels
    layers.append(nn.MaxPool2d(2))
    return nn.Sequential(*layers)


class MiniVGG(nn.Module):
    """Три правила VGG на нашому масштабі: 3×3, пулінг 2×2, подвоєння каналів."""

    def __init__(self, num_classes=6):
        super().__init__()
        self.features = nn.Sequential(
            vgg_block(1, 8, convs=2),      # 1×28×28  →  8×14×14
            vgg_block(8, 16, convs=2),     # 8×14×14  → 16×7×7
            vgg_block(16, 32, convs=2),    # 16×7×7   → 32×3×3
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 3 * 3, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


torch.manual_seed(0)
mini_vgg = MiniVGG()

mini_total = sum(p.numel() for p in mini_vgg.parameters())
conv_weights = sum(p.numel() for m in mini_vgg.modules() if isinstance(m, nn.Conv2d)
                   for p in m.parameters(recurse=False))
head_weights = sum(p.numel() for m in mini_vgg.modules() if isinstance(m, nn.Linear)
                   for p in m.parameters(recurse=False))
norm_weights = mini_total - conv_weights - head_weights
first_linear_weights = 32 * 3 * 3 * 64 + 64

print("MiniVGG: шість згорток 3×3, три пулінги, голова з двох Linear")
print(f"  у згортках          : {conv_weights:>7,}  ({100*conv_weights/mini_total:.1f} %)")
print(f"  у батчнормі         : {norm_weights:>7,}  ({100*norm_weights/mini_total:.1f} %)")
print(f"  у голові            : {head_weights:>7,}  ({100*head_weights/mini_total:.1f} %)")
print(f"  з них Linear(288→64): {first_linear_weights:>7,}  "
      f"({100*first_linear_weights/mini_total:.1f} %)")
print(f"  усього              : {mini_total:>7,}")
print()
print("Та сама картина, що у справжньої VGG16: більшість ваг — у голові,")
print("і майже всі вони — в одному шарі одразу після Flatten.")

In [ ]:
def train_once(model, x, y, epochs=10, batch_size=64, learning_rate=3e-2):
    """Одне навчання: SGD із моментом, перемішування прикладів кожної епохи."""
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    loss_fn = nn.CrossEntropyLoss()
    model.train()
    for epoch in range(epochs):
        order = torch.randperm(len(x))
        running_loss = 0.0
        for start in range(0, len(x), batch_size):
            batch = order[start:start + batch_size]
            optimizer.zero_grad()
            loss = loss_fn(model(x[batch]), y[batch])
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * len(batch)
        print(f"  епоха {epoch+1:>2}: втрата {running_loss/len(x):.4f}")
    return model


def accuracy(model, x, y):
    model.eval()
    with torch.no_grad():
        predicted = model(x).argmax(dim=1)
    return (predicted == y).float().mean().item()


torch.manual_seed(0)
started = time.perf_counter()
train_once(mini_vgg, train_x, train_y)
print()
print(f"навчання зайняло {time.perf_counter()-started:.1f} с")
print(f"точність на перевірці: {accuracy(mini_vgg, test_x, test_y):.3f}"
      f"   (рівень вгадування на шести класах — 0.167)")

### І та сама мережа з головою на глобальному усередненні

Замінюємо `Flatten` + два `Linear` на глобальне усереднення й один `Linear`. Це те
саме, що зробили з VGG16 наступні архітектури — тільки в мініатюрі.

In [ ]:
class MiniVGGWithGAP(nn.Module):
    """Те саме тіло, але голова — глобальне усереднення замість Flatten."""

    def __init__(self, num_classes=6):
        super().__init__()
        self.features = nn.Sequential(
            vgg_block(1, 8, convs=2),
            vgg_block(8, 16, convs=2),
            vgg_block(16, 32, convs=2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),      # від кожного каналу лишається одне число
            nn.Flatten(),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


torch.manual_seed(0)
gap_model = MiniVGGWithGAP()
gap_total = sum(p.numel() for p in gap_model.parameters())

print(f"MiniVGG із Flatten: {mini_total:>7,} параметрів")
print(f"MiniVGG із GAP    : {gap_total:>7,} параметрів")
print(f"схудла у {mini_total/gap_total:.1f} раза")
print()

# і головне: GAP приймає вхід будь-якого розміру, а Flatten — лише один
for side in (20, 28, 44):
    sample = torch.zeros(1, 1, side, side)
    with torch.no_grad():
        gap_output = gap_model(sample)
    try:
        with torch.no_grad():
            mini_vgg(sample)
        flatten_result = "працює"
    except RuntimeError:
        flatten_result = "падає"
    print(f"вхід {side}×{side}: GAP {tuple(gap_output.shape)}, Flatten — {flatten_result}")

## Завдання

### 🟢 Рівень 1
Побудуй `models.vgg11(weights=None)` і `models.vgg19(weights=None)` і надрукуй для них
ту саму таблицю, що для VGG16: усього параметрів, у згортках, у голові. Скільки
відсотків ваги додають вісім зайвих згорток?

### 🟡 Рівень 2
Прибери `nn.BatchNorm2d` із `vgg_block`, навчи мережу ще раз тими самими
налаштуваннями й порівняй точність. Поясни словами, чому шість згорток поспіль без
нормалізації навчаються гірше за три згортки з теми 09.

### 🔴 Рівень 3
Порахуй, у скільки разів схудне VGG16, якщо замінити голову на глобальне усереднення
плюс один `Linear` на тисячу класів, — і **доведи розрахунком**: окремо покажи ваги
згорткової частини, ваги старої голови, ваги нової голови й підсумок. Перевір, що
твоя нова мережа приймає вхід іншого розміру, а стара — ні.